idea da https://chatgpt.com/share/6a69a13a-9354-83eb-acb6-fc1e20d4d001?ogimg=plain

In [1]:
import random as rnd
import numpy as np

In [2]:
class Firm():
    def __init__(self, uid: int, country: int, sector: int, targetAddedValue:float, outputWarehouse: float):
        self.uid = uid
        self.country = country
        self.sector = sector
        self.targetAddedValue = targetAddedValue
        self.outputWarehouse = outputWarehouse
    def info(self):
        print(self.uid, "c = ", self.country, "s = ",self.sector, "targetAV =",self.targetAddedValue)

In [3]:
rnd.seed(12345)
allFirms=[]
#setup firms
uid = 0
for sector in range(64):
    for country in range(27):
        for n in range(rnd.randint(100,2000)):
            targetAddedValue = rnd.uniform(10000, 100000)
            outputWarehouse = rnd.uniform(10000, 100000)
            aFirm = Firm(uid, country, sector, targetAddedValue, outputWarehouse)
            uid += 1
            allFirms.append(aFirm)
print (uid, len(allFirms))

1786416 1786416


In [4]:
aFirm.info()

1786415 c =  26 s =  63 targetAV = 89286.56256164442


In [5]:
#collecting firms

collection = [[[] for _ in range(27)] for _ in range(64)]

for aFirm in allFirms:
    collection [aFirm.sector][aFirm.country].append(aFirm)

In [6]:
sum(len(c) for s in collection for c in s)

1786416

In [7]:
def match_buyers_to_suppliers(
    buyers,
    suppliers,
    buyer_country: int,
    k: float,
    rng: np.random.Generator,
    point_mode: str = "systematic",
):
    """
    Matches the buyers of a country with the suppliers of a sector
    through two aligned tapes.

    The buyer does not directly store the reference to the supplier,
    but the supplier's position in the permuted list.

    Parameters
    ----------
    buyers
        List of buyers belonging to the same country.

    suppliers
        List of suppliers belonging to the same sector.

    buyer_country
        Country of the buyers, between 0 and 26.

    k
        Weight multiplier for suppliers belonging to the same country.

    rng
        NumPy random generator, for example:

        rng = np.random.default_rng(12345)

    point_mode
        "starts"
            Uses the exact starting point of each buyer's segment.

        "systematic"
            Uses equally spaced points with a common random shift.

        "stratified"
            Uses a random point within each buyer's segment.

    Returns
    -------
    permuted_suppliers
        List of suppliers in the order of the permuted tape.

    matched_positions
        Integer array. For each buyer, it contains the position
        of the assigned supplier in the permuted_suppliers list.

    permutation
        Array connecting the permuted list to the original list:

        permuted_suppliers[p] is suppliers[permutation[p]]
    """

    n_buyers = len(buyers)
    n_suppliers = len(suppliers)

    if n_buyers == 0:
        return (
            [],
            np.empty(0, dtype=np.int64),
            np.empty(0, dtype=np.int64),
        )

    if n_suppliers == 0:
        raise ValueError("La lista dei supplier è vuota.")

    if k <= 0:
        raise ValueError("k deve essere strettamente positivo.")

    if not 0 <= buyer_country <= 26:
        raise ValueError("buyer_country deve essere compreso tra 0 e 26.")

    # ------------------------------------------------------------
    # 1. Extract supplier data
    # ------------------------------------------------------------

    supplier_countries = np.fromiter(
        (supplier.country for supplier in suppliers),
        dtype=np.int16,
        count=n_suppliers,
    )

    added_values = np.fromiter(
        (supplier.targetAddedValue for supplier in suppliers),
        dtype=np.float64,
        count=n_suppliers,
    )

    if np.any(~np.isfinite(added_values)):
        raise ValueError(
            "targetAddedValue contiene valori non finiti."
        )

    if np.any(added_values < 0):
        raise ValueError(
            "targetAddedValue contiene valori negativi."
        )

    # ------------------------------------------------------------
    # 2. Build the weights for the country-specific version
    #    of tape B
    # ------------------------------------------------------------

    weights = added_values.copy()

    same_country = supplier_countries == buyer_country
    weights[same_country] *= k

    total_weight = weights.sum()

    if total_weight <= 0:
        raise ValueError(
            "La somma dei pesi dei supplier deve essere positiva."
        )

    # ------------------------------------------------------------
    # 3. Shuffle tape B
    # ------------------------------------------------------------

    permutation = rng.permutation(n_suppliers)

    permuted_suppliers = [
        suppliers[original_position]
        for original_position in permutation
    ]

    permuted_weights = weights[permutation]

    # ------------------------------------------------------------
    # 4. Right endpoints of the intervals occupied by suppliers
    # ------------------------------------------------------------

    cumulative_weights = np.cumsum(permuted_weights)

    # Avoid small numerical issues at the final endpoint.
    cumulative_weights[-1] = total_weight

    # ------------------------------------------------------------
    # 5. Buyer positions on tape A
    # ------------------------------------------------------------

    buyer_number = np.arange(
        n_buyers,
        dtype=np.float64,
    )

    if point_mode == "starts":

        buyer_points = (
            buyer_number
            * total_weight
            / n_buyers
        )

    elif point_mode == "systematic":

        # A single random shift for the entire tape A.
        offset = rng.random()

        buyer_points = (
            buyer_number + offset
        ) * total_weight / n_buyers

    elif point_mode == "stratified":

        # A random point within each buyer's segment.
        buyer_points = (
            buyer_number + rng.random(n_buyers)
        ) * total_weight / n_buyers

    else:
        raise ValueError(
            "point_mode deve essere 'starts', "
            "'systematic' oppure 'stratified'."
        )

    # ------------------------------------------------------------
    # 6. Search for the supplier interval containing each point
    # ------------------------------------------------------------

    matched_positions = np.searchsorted(
        cumulative_weights,
        buyer_points,
        side="right",
    ).astype(np.int64)

    return (
        permuted_suppliers,
        matched_positions,
        permutation,
    )

In [8]:
buyers_by_country=[[]]*27
for country_id in range(27):
    buyers_by_country[country_id] = [firm for sector in collection for firm in sector[country_id]]

In [9]:
#len(buyers_by_country[0])

In [10]:
suppliers_by_sector=[[]]*64
for sector_id in range(64):
    suppliers_by_sector[sector_id] = [firm for country in collection[sector_id] for firm in country]

In [11]:
#len(suppliers_by_sector[0])

In [12]:
rng = np.random.default_rng(12345)

suppliers = suppliers_by_sector[0]
buyers = buyers_by_country[0]

(
    permuted_suppliers,
    matched_positions,
    permutation,
) = match_buyers_to_suppliers(
    buyers=buyers,
    suppliers=suppliers,
    buyer_country=0,
    k=2.0,
    rng=rng,
    point_mode="systematic",
)

In [13]:
for buyer, position in zip(
    buyers,
    matched_positions,
):
    buyer.supplierPosition = int(position)

In [14]:
position=buyers_by_country[0][32].supplierPosition
position

13

In [15]:
supplier = permuted_suppliers[position]

In [16]:
supplier.outputWarehouse

73569.39942263722

In [17]:
right_position = (
    position + 1
) % len(permuted_suppliers)

right_supplier = permuted_suppliers[right_position]

In [18]:
right_position

14

In [19]:
right_supplier.outputWarehouse

13145.882755539216

In [20]:
supplier.uid

6035

In [21]:
for aFirm in allFirms:
    if aFirm.uid==6035: print(aFirm.country, aFirm.sector, aFirm.outputWarehouse)

7 0 73569.39942263722
